# Example Notebook: Stock market returns and ruling party

This notebook shows how we can translate a research question into a testable hypothesis, find data and test it empirically.

### 1. Research Question:
Does the stock market perform better when Republicans are in office?


### 2. Testable hypothesis:
When the US has a Republican president, real stock market returns tend to be higher than when a Democrat is in office.

In [ ]:
# Load necessary libraries
import statsmodels.api as sm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import fredapi


### 3. Finding Data using FRED
We need to find data on **real stock market returns** and an indicator of the **president's party affiliation**.

A return is the change in the share price **plus the dividends paid**. FRED has share prices, but no dividends on the same scale, so we use a **dividend yield**: what US nonfinancial corporations pay in dividends, divided by the market value of their shares. Both come from the Federal Reserve's Financial Accounts.


In [ ]:
# Set up FRED data connection
# Instructions for getting your FRED API key available at these websites
# https://pypi.org/project/fredapi/ 
# http://api.stlouisfed.org/api_key.html
from fredapi import Fred
# The key is read from the FRED_API_KEY environment variable; never paste it here.
fred = Fred()

# One sample for every series: monthly, from 1960 to July 2023, the last month of the GOP flags below
START, END = '1960-01-01', '2023-07-01'

# OECD Total share price index (monthly, 2015 = 100)
temp = fred.get_series('SPASTT01USM661N', observation_start=START, observation_end=END)
df_total_share_price_idx = pd.DataFrame(data = temp, columns=['Total_Price_Index'])
df_total_share_price_idx = df_total_share_price_idx.rename_axis('Date').reset_index()
print(df_total_share_price_idx.head(5))

# Dividend yield, quarterly: US nonfinancial corporations' net dividends paid
# ($ millions, at an annual rate) over the market value of their shares ($ millions, end of quarter)
div = fred.get_series('BOGZ1FA106121075Q', observation_start='1959-10-01', observation_end=END)
mv = fred.get_series('NCBEILQ027S', observation_start='1959-10-01', observation_end=END)
# A quarter's dividends, at a monthly rate, over the value of the shares at the start of the quarter
df_dividends = pd.DataFrame({'Dividend_Yield': (div / 12) / mv.shift()}).rename_axis('Quarter').reset_index()
print(df_dividends.head(5))

# Load CPI (monthly), and the factor that puts prices in July 2023 dollars
temp = fred.get_series('CPIAUCSL', observation_start=START, observation_end=END)
df_cpi = pd.DataFrame(data = temp, columns=['CPI'])
df_cpi = df_cpi.rename_axis('Date').reset_index()

cpi_base = df_cpi.loc[df_cpi["Date"] == END, "CPI"].item()
df_cpi['CPI_ADJ'] = cpi_base / df_cpi['CPI']
print(df_cpi.head(5))


In [ ]:
# Combine FRED data and define the real return for each month
df_fred = pd.merge(df_total_share_price_idx, df_cpi, on='Date')
df_fred['Quarter'] = df_fred['Date'].dt.to_period('Q').dt.start_time
df_fred = pd.merge(df_fred, df_dividends, on='Quarter').sort_values('Date').reset_index(drop=True)
# Check: one row per month, with none missing
assert len(df_fred) == len(pd.date_range(START, END, freq='MS'))

# Dividends paid in the month, on the share-price index's scale: the monthly yield times last month's price
df_fred['Total_Dividends_Index'] = df_fred['Dividend_Yield'] * df_fred['Total_Price_Index'].shift()

df_fred['Real_Price_Index'] = df_fred['Total_Price_Index']*df_fred['CPI_ADJ']
df_fred['Real_Dividends_Index'] = df_fred['Total_Dividends_Index']*df_fred['CPI_ADJ']
df_fred['Real_Price_Index_Lag'] = df_fred['Real_Price_Index'].shift()

df_fred['Real_Month_Return'] = (df_fred['Real_Price_Index'] + df_fred['Real_Dividends_Index']) / df_fred['Real_Price_Index_Lag']
# What a dollar invested in January 1960 grows to, dividends reinvested, in July 2023 dollars
df_fred['Real_Total_Return'] = df_fred['Real_Price_Index'].iloc[0] * df_fred['Real_Month_Return'].fillna(1).cumprod()
df_fred['log_Real_Total_Return'] = np.log(df_fred['Real_Total_Return'])
df_fred['log_Real_Month_Return'] = np.log(df_fred['Real_Month_Return'])
print(df_fred)


In [ ]:
# Read in data files 
# GOP term flags
url = "../presidential-term-data/GOP_flag.csv"  # in this repository
temp = pd.read_csv(url,dtype=str)
df_gop = pd.DataFrame(temp)

# The file stores each month as a decimal number, YYYY.MM, so October lost its trailing zero:
# it is written 1960.1, not 1960.10, and reading it as a date makes it a second January.
# Split year from month and pad the month on the RIGHT: '1' becomes '10'; '01'...'12' are unchanged.
parts = df_gop['Date'].str.split('.', expand=True)
month = parts[1].str.ljust(2, '0')
df_gop['Date'] = pd.to_datetime(parts[0] + '-' + month, format='%Y-%m')  # first day of the month
df_gop['GOP'] = df_gop['GOP'].astype(int).astype(bool)  # True in a month with a Republican president

# Check: one row per month, with none repeated and none missing
assert df_gop['Date'].is_unique
assert len(df_gop) == len(pd.date_range(df_gop['Date'].min(), df_gop['Date'].max(), freq='MS'))
print(df_gop.head())

In [ ]:
# Create new data frame that merges GOP flag with returns data
df = pd.merge(df_fred, df_gop, left_on='Date', right_on='Date')
print(df.head(5))

In [ ]:
# Plot real total returns (log scale). Shade GOP presidential terms
# convenience function, used for both plots
def plot_series(ax, df, index='Date', cols=['log_Real_Total_Return'], area='GOP'):
    # set up an index based on date
    df = df.set_index(keys=index, drop=False)
    # line plot
    df.plot(ax=ax, x=index, y=cols, color='blue')
    # extract limits
    y1, y2 = ax.get_ylim()
    ax.fill_between(df[index].index, y1=y1, y2=y2, where=df[area], facecolor='grey', alpha=0.4)
    return ax

# set up figure, axis
f, ax = plt.subplots()
plot_series(ax, df)
ax.grid(True)
plt.show()


In [ ]:
# Plot monthly log real returns over time. Shade GOP presidential terms
f, ax = plt.subplots()
plot_series(ax, df, cols=['log_Real_Month_Return'])
ax.grid(True)
plt.show()


### 4. Testing hypothesis empirically with OLS

In [ ]:
# Run OLS regression of monthly real returns onto the GOP president flag
# The flag is lagged one month: the party in office is known before the month's return.
df['GOPlag'] = df['GOP'].shift()

results = sm.formula.ols(formula = 'log_Real_Month_Return ~ GOPlag', data=df).fit(cov_type='HC3')
print(results.summary())


### 5. Conclusion
The regression analysis shows that the stock market performs worse under a GOP presidency.